In [1]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table, read_table

ModuleNotFoundError: No module named 'utils'

In [ ]:
df = pd.read_csv("../../data/state_graduates/Johor.csv")
df.head(20)

In [ ]:
def transform_graduate_state_data(file_path):
    df = pd.read_csv(file_path)

    # reshape years into rows
    df_long = df.melt(
        id_vars=["state", "statistics"],
        var_name="year",
        value_name="value"
    )

    # extract metric and qualification from statistics
    def parse_stat(stat):
        if pd.isna(stat):
            return pd.Series([None, None], index=['metric', 'qualification'])
        parts = str(stat).rsplit('_', 1)
        if len(parts) == 2 and parts[1] in ['degree', 'diploma']:
            return pd.Series([parts[0], parts[1]], index=['metric', 'qualification'])
        return pd.Series([stat, None], index=['metric', 'qualification'])

    df_long[['metric', 'qualification']] = df_long['statistics'].apply(parse_stat)

    # pivot metrics into columns (one row per state/year/qualification)
    df_pivot = df_long.pivot_table(
        index=['state', 'year', 'qualification'],
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()

    df_pivot.columns.name = None

    num_cols = [c for c in df_pivot.columns if c not in ['state', 'year', 'qualification']]
    for col in num_cols:
        df_pivot[col] = pd.to_numeric(df_pivot[col], errors='coerce')
        if col.endswith('_rate'):
            continue
        df_pivot[col] = df_pivot[col] * 1000

    df_pivot = df_pivot.sort_values(['state', 'year', 'qualification']).reset_index(drop=True)
    return df_pivot

In [ ]:
df_johor = transform_graduate_state_data("../../data/state_graduates/Johor.csv")
df_johor.head(10)

In [ ]:
df_kedah = transform_graduate_state_data("../../data/state_graduates/Kedah.csv")
df_kedah.head(10)

In [ ]:
df_kelantan = transform_graduate_state_data("../../data/state_graduates/Kelantan.csv")
df_kelantan.head(10)

In [ ]:
df_kl = transform_graduate_state_data("../../data/state_graduates/Kuala Lumpur.csv")
df_kl.head(10)

In [ ]:
df_labuan = transform_graduate_state_data("../../data/state_graduates/Labuan.csv")
df_labuan.head(10)

In [ ]:
df_melaka = transform_graduate_state_data("../../data/state_graduates/Melaka.csv")
df_melaka.head(10)

In [ ]:
df_ns = transform_graduate_state_data("../../data/state_graduates/Negeri Sembilan.csv")
df_ns.head(10)

In [ ]:
df_pahang = transform_graduate_state_data("../../data/state_graduates/Pahang.csv")
df_pahang.head(10)

In [ ]:
df_penang = transform_graduate_state_data("../../data/state_graduates/Pulau Pinang.csv")
df_penang.head(10)

In [ ]:
df_perak = transform_graduate_state_data("../../data/state_graduates/Perak.csv")
df_perak.head(10)

In [ ]:
df_perlis = transform_graduate_state_data("../../data/state_graduates/Perlis.csv")
df_perlis.head(10)

In [ ]:
df_putrajaya = transform_graduate_state_data("../../data/state_graduates/Putrajaya.csv")
df_putrajaya.head(10)

In [ ]:
df_sabah = transform_graduate_state_data("../../data/state_graduates/Sabah.csv")
df_sabah.head(10)

In [ ]:
df_sarawak = transform_graduate_state_data("../../data/state_graduates/Sarawak.csv")
df_sarawak.head(10)

In [ ]:
df_selangor = transform_graduate_state_data("../../data/state_graduates/Selangor.csv")
df_selangor.head(10)

In [ ]:
df_terengganu = transform_graduate_state_data("../../data/state_graduates/Terengganu.csv")
df_terengganu.head(10)

In [ ]:
df_merged = pd.concat([
    df_johor, df_kedah, df_kelantan, df_kl, df_labuan, df_melaka, df_ns, df_pahang, df_penang, df_perak, df_perlis,
    df_putrajaya, df_sabah, df_sarawak, df_selangor, df_terengganu
], ignore_index=True)
write_table(df_merged, "sc_bronze", "dosm_graduates_state")
df_merged.head(10)